In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Szacowany czas użycia: 16 minut na procesorze Eagle r3 (UWAGA: To jest jedynie szacunek. Rzeczywisty czas może się różnić.)*

In [ ]:
# This cell is hidden from users;
# it disables linting rules.
# ruff: noqa

## Tło

Wsteczna propagacja operatora to technika polegająca na wchłanianiu operacji z końca Circuit kwantowego do mierzonej obserwowalnej, co generalnie zmniejsza głębokość Circuit kosztem dodatkowych składników obserwowalnej. Celem jest wsteczna propagacja możliwie dużej części Circuit bez dopuszczenia do nadmiernego wzrostu obserwowalnej. Implementacja oparta na Qiskit jest dostępna w dodatku OBP Qiskit, więcej szczegółów można znaleźć w odpowiedniej [dokumentacji](/guides/qiskit-addons-obp) wraz z [prostym przykładem](/guides/qiskit-addons-obp-get-started), od którego można zacząć.

Rozważmy przykładowy Circuit, dla którego ma zostać zmierzona obserwowalna $O = \sum_P c_P P$, gdzie $P$ to operatory Pauliego, a $c_P$ to współczynniki. Oznaczmy Circuit jako pojedynczą unitarność $U$, którą można logicznie podzielić na $U = U_C U_Q$, jak pokazano na poniższym rysunku.

![Diagram Circuit pokazujący Uq a następnie Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Wsteczna propagacja operatora wchłania unitarność $U_C$ do obserwowalnej poprzez jej ewolucję jako $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Innymi słowy, część obliczeń jest wykonywana klasycznie przez ewolucję obserwowalnej od $O$ do $O'$. Oryginalny problem można teraz przeformułować jako pomiar obserwowalnej $O'$ dla nowego Circuit o mniejszej głębokości, którego unitarność wynosi $U_Q$.

Unitarność $U_C$ jest reprezentowana przez pewną liczbę plasterków $U_C = U_S U_{S-1}...U_2U_1$. Istnieje wiele sposobów definiowania plasterka. Na przykład w powyższym przykładowym Circuit każda warstwa bramek $R_{zz}$ i każda warstwa bramek $R_x$ może być traktowana jako osobny plasterek. Wsteczna propagacja polega na klasycznym obliczaniu $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$. Każdy plasterek $U_s$ można reprezentować jako $U_s = exp(\frac{-i\theta_s P_s}{2})$, gdzie $P_s$ jest $n$-qubitowym operatorem Pauliego, a $\theta_s$ jest skalarem. Łatwo wykazać, że

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$
$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

W powyższym przykładzie, jeśli ${P,P_s} = 0$, to zamiast jednego Circuit musimy wykonać dwa Circuit kwantowe, aby obliczyć wartość oczekiwaną. Dlatego wsteczna propagacja może zwiększać liczbę składników obserwowalnej, prowadząc do większej liczby uruchomień Circuit. Jednym ze sposobów na umożliwienie głębszej wstecznej propagacji do Circuit przy jednoczesnym zapobieganiu nadmiernemu wzrostowi operatora jest obcinanie składników o małych współczynnikach zamiast dodawania ich do operatora. Na przykład w powyższym przykładzie można zdecydować się na obcięcie składnika zawierającego $P_sP$, pod warunkiem że $\theta_s$ jest wystarczająco małe. Obcinanie składników może skutkować mniejszą liczbą Circuit kwantowych do wykonania, ale powoduje pewien błąd w końcowym obliczeniu wartości oczekiwanej proporcjonalny do wartości bezwzględnych współczynników obciętych składników.

Ten samouczek implementuje [wzorzec Qiskit](/guides/intro-to-patterns) do symulacji dynamiki kwantowej łańcucha spinów Heisenberga z użyciem <a href="https://github.com/Qiskit/qiskit-addon-obp">qiskit-addon-obp</a>.

## Wymagania

Przed rozpoczęciem tego samouczka upewnij się, że masz zainstalowane następujące pakiety:

- Qiskit SDK v1.2 lub nowszy (`pip install qiskit`)
- Qiskit Runtime v0.28 lub nowszy (`pip install qiskit-ibm-runtime`)
- Dodatek OBP Qiskit (`pip install qiskit-addon-obp`)
- Narzędzia pomocnicze Qiskit addon (`pip install qiskit-addon-utils`)

## Konfiguracja

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_gate_types, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Część I: Łańcuch spinów Heisenberga w małej skali
### Krok 1: Odwzorowanie danych wejściowych na problem kwantowy
#### Odwzorowanie ewolucji czasowej kwantowego modelu Heisenberga na eksperyment kwantowy.
Pakiet [qiskit_addon_utils](https://github.com/Qiskit/qiskit-addon-utils) dostarcza pewnych funkcjonalności wielokrotnego użytku w różnych celach.

Jego moduł [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) udostępnia funkcje do generowania hamiltonianów podobnych do Heisenberga na danym grafie połączeń.
Graf ten może być zarówno [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html), jak i [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap), co ułatwia korzystanie z niego w przepływach pracy zorientowanych na Qiskit.

Poniżej generujemy liniowy łańcuch `CouplingMap` złożony z 10 Qubitów.

In [3]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/de8ce35e-a2c5-474f-adb9-88c9afb2bd76-0.avif)

Następnie generujemy operator Pauliego modelujący hamiltonian Heisenberga XYZ.

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

Gdzie $G(V,E)$ jest grafem dostarczonej mapy sprzężeń.

In [4]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

From the qubit operator, we can generate a quantum circuit which models its time evolution.
Once again, the [qiskit_addon_utils.problem_generators](/docs/api/qiskit-addon-utils/problem-generators) module comes to the rescue with a handy function do just that:

In [5]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", scale=0.6)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif" alt="Output of the previous code cell" />

Na podstawie operatora qubitowego możemy wygenerować Circuit kwantowy modelujący jego ewolucję czasową.
Ponownie z pomocą przychodzi moduł [qiskit_addon_utils.problem_generators](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators) z przydatną funkcją do właśnie tego celu:

In [6]:
slices = slice_by_gate_types(circuit)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


![Wynik poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/1d68f197-ffa4-49de-9fe8-243b1facbd00-0.avif)

### Krok 2: Optymalizacja problemu pod kątem wykonania na sprzęcie kwantowym
#### Tworzenie plasterków Circuit do wstecznej propagacji
Pamiętaj, że funkcja ``backpropagate`` propaguje wstecznie całe plasterki Circuit naraz, więc wybór sposobu podziału może mieć wpływ na efektywność wstecznej propagacji dla danego problemu. Tutaj pogrupujemy bramki tego samego typu w plasterki, używając funkcji [slice_by_gate_types](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_gate_types).

Aby uzyskać bardziej szczegółowe omówienie podziału Circuit na plasterki, zapoznaj się z tym [przewodnikiem how-to](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) pakietu [qiskit-addon-utils](https://github.com/Qiskit/qiskit-addon-utils).

In [7]:
op_budget = OperatorBudget(max_qwc_groups=8)

#### Backpropagate slices from the circuit

First we specify the observable to be $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, $N$ being the number of qubits. We will backpropagate slices from the time-evolution circuit until the terms in the observable can no longer be combined into eight or fewer qubit-wise commuting Pauli groups.

In [8]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

#### Ograniczanie rozmiaru, do jakiego operator może urosnąć podczas wstecznej propagacji
Podczas wstecznej propagacji liczba składników operatora będzie generalnie szybko zbliżać się do $4^N$, gdzie $N$ to liczba Qubitów. Gdy dwa składniki operatora nie komutują qubit po qubicie, potrzebujemy osobnych Circuit do uzyskania odpowiadających im wartości oczekiwanych. Na przykład, jeśli mamy 2-qubitową obserwowalną $O = 0.1 XX + 0.3 IZ - 0.5 IX$, to ponieważ $[XX,IX] = 0$, pomiar w jednej bazie wystarczy do obliczenia wartości oczekiwanych dla tych dwóch składników. Jednak $IZ$ antykomutuje z pozostałymi dwoma składnikami. Potrzebujemy więc osobnego pomiaru bazowego do obliczenia wartości oczekiwanej $IZ$. Innymi słowy, do obliczenia $\langle O \rangle$ potrzebujemy dwóch Circuit zamiast jednego. Wraz ze wzrostem liczby składników operatora istnieje możliwość, że wymagana liczba uruchomień Circuit również wzrośnie.

Rozmiar operatora można ograniczyć, podając argument ``operator_budget`` funkcji ``backpropagate``, który przyjmuje instancję [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Aby kontrolować ilość dodatkowych zasobów (czasu) przydzielonych, ograniczamy maksymalną liczbę qubitowo komutujących grup Pauliego, jaką może mieć propagowana wstecznie obserwowalna. Tutaj określamy, że wsteczna propagacja powinna zatrzymać się, gdy liczba qubitowo komutujących grup Pauliego w operatorze przekroczy 8.

In [9]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/65ec9cb1-a4ed-497b-a616-180e9659956f-1.avif" alt="Output of the previous code cell" />

#### Wsteczna propagacja plasterków z Circuit
Najpierw określamy obserwowalną jako $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, gdzie $N$ to liczba Qubitów. Będziemy propagować wstecznie plasterki z Circuit ewolucji czasowej, dopóki składniki obserwowalnej będą mogły być łączone w osiem lub mniej qubitowo komutujących grup Pauliego.

In [10]:
truncation_error_budget = setup_budget(max_error_per_slice=0.005)

Note that by allocating `5e-3` error per slice for truncation, we are able to remove 1 more slice from the circuit, while remaining within the original budget of eight commuting Pauli groups in the observable. By default, `backpropagate` uses the L1 norm of the truncated coefficients to bound the total error incurred from truncation. For other options refer to the [how-to guide on specifying the p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

In this particular example where we have backpropagated seven slices, the total truncation error should not exceed ``(5e-3 error/slice) * (7 slices) = 3.5e-2``.
For further discussion on distributing an error budget across your slices, check out [this how-to guide](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [11]:
# Run the same experiment but truncate observable terms with small coefficients
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)

# Recombine the slices remaining after backpropagation
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs_trunc.paulis)} terms, which can be combined into {len(bp_obs_trunc.group_commuting(qubit_wise=True))} groups.\n"
    f"After truncation, the error in our observable is bounded by {metadata.accumulated_error(0):.3e}"
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit_trunc.draw("mpl", scale=0.6)

Backpropagated 7 slices.
New observable has 82 terms, which can be combined into 8 groups.
After truncation, the error in our observable is bounded by 3.266e-02
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5e8bae1a-ef18-4eb0-9d2a-1ac7bbdced3b-1.avif" alt="Output of the previous code cell" />

Poniżej zobaczysz, że propagowaliśmy wstecznie sześć plasterków, a składniki zostały połączone w sześć, a nie osiem grup. Oznacza to, że propagacja wsteczna jednego kolejnego plasterka spowodowałaby przekroczenie liczby ośmiu grup Pauliego. Możemy to potwierdzić, sprawdzając zwrócone metadane. Zwróć też uwagę, że w tej części transformacja Circuit jest dokładna. To znaczy, że żadne składniki nowej obserwowalnej $O'$ nie zostały obcięte. Propagowany wstecznie Circuit i propagowana wstecznie obserwowalna dają dokładnie taki sam wynik jak oryginalny Circuit i obserwowalna.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

In [13]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

# Transpile original experiment
circuit_isa = pm.run(circuit)
observable_isa = observable.apply_layout(circuit_isa.layout)

# Transpile backpropagated experiment
bp_circuit_isa = pm.run(bp_circuit)
bp_obs_isa = bp_obs.apply_layout(bp_circuit_isa.layout)

# Transpile the backpropagated experiment with truncated observable terms
bp_circuit_trunc_isa = pm.run(bp_circuit_trunc)
bp_obs_trunc_isa = bp_obs_trunc.apply_layout(bp_circuit_trunc_isa.layout)

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/65ec9cb1-a4ed-497b-a616-180e9659956f-1.avif)

Następnie określimy ten sam problem z takimi samymi ograniczeniami na rozmiar wyjściowej obserwowalnej. Tym razem jednak przydzielimy budżet błędu dla każdego plasterka, używając funkcji [setup_budget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-truncating#setup_budget). Składniki Pauliego o małych współczynnikach będą obcinane z każdego plasterka, dopóki budżet błędu nie zostanie wyczerpany, a pozostały budżet zostanie dodany do budżetu następnego plasterka. Zwróć uwagę, że w tym przypadku transformacja wynikająca z wstecznej propagacji jest przybliżona, ponieważ niektóre składniki operatora są obcinane.

Aby włączyć to obcinanie, musimy skonfigurować nasz budżet błędu w następujący sposób:

In [14]:
pub = (circuit_isa, observable_isa)
bp_pub = (bp_circuit_isa, bp_obs_isa)
bp_trunc_pub = (bp_circuit_trunc_isa, bp_obs_trunc_isa)

Zwróć uwagę, że przydzielając błąd `5e-3` na plasterek do obcinania, jesteśmy w stanie usunąć 1 dodatkowy plasterek z Circuit, pozostając w ramach pierwotnego budżetu ośmiu komutujących grup Pauliego w obserwowalnej. Domyślnie `backpropagate` używa normy L1 obciętych współczynników do ograniczenia całkowitego błędu wynikającego z obcinania. Inne opcje znajdziesz w [przewodniku how-to dotyczącym określania p_norm](https://qiskit.github.io/qiskit-addon-obp/how_tos/bound_error_using_p_norm.html).

W tym konkretnym przykładzie, gdzie propagowaliśmy wstecznie siedem plasterków, całkowity błąd obcinania nie powinien przekroczyć ``(5e-3 error/slice) * (7 slices) = 3.5e-2``.
Więcej informacji na temat dystrybucji budżetu błędu między plasterkami znajdziesz w [tym przewodniku how-to](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html).

In [15]:
ideal_estimator = Estimator()

# Run the experiments using Estimator primitive to obtain the exact outcome
result_exact = (
    ideal_estimator.run([(circuit, observable)]).result()[0].data.evs.item()
)
print(f"Exact expectation value: {result_exact}")

Exact expectation value: 0.8871244838989416


We shall use <a href="/docs/guides/configure-error-mitigation">resilience_level</a> = 2 for this example.

In [ ]:
options = EstimatorOptions()
options.default_precision = 0.011
options.resilience_level = 2

estimator = EstimatorV2(mode=backend, options=options)

In [ ]:
job = estimator.run([pub, bp_pub, bp_trunc_pub])

### Step 4: Post-process and return result to desired classical format

In [ ]:
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

In [ ]:
print(
    f"Expectation value without backpropagation: {result_no_bp} ± {std_no_bp}"
)
print(f"Backpropagated expectation value: {result_bp} ± {std_bp}")
print(
    f"Backpropagated expectation value with truncation: {result_bp_trunc} ± {std_bp_trunc}"
)

Expectation value without backpropagation: 0.8033194665993642
Backpropagated expectation value: 0.8599808781259016
Backpropagated expectation value with truncation: 0.8868736004169483


In [ ]:
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
stds = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(result_exact)
ax.set_ylim([0.6, 0.92])
plt.text(0.2, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/b444d8bc-c800-4aa3-9927-eb807e92194f-1.avif" alt="Output of the previous code cell" />

## Part B: Scale it up!

Let us now use Operator Backpropagation to study the dynamics of the Hamiltonian of a 50-qubit Heisenberg Spin Chain.

### Step 1: Map classical inputs to a quantum problem

We consider a 50-qubit Hamiltonian $\hat{\mathcal{H}}_{XYZ}$ for the scaled up problem with the same values for the $J$ and $h$ coefficients as in the small-scale example. The observable $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$ is also the same as before. This problem is beyond classical brute-force simulation.

In [16]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/47cb1ac7-44db-4f96-b49b-e889a920d83c-0.avif" alt="Output of the previous code cell" />

In [17]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXXIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIYYIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZZIIIIIIIIIII', 'IIIIIIIIIIII

In [18]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIIIIIIIIIIIIIIII', 'IIIIIIIIIIII

### Krok 4: Post-processing i zwrócenie wyniku w żądanym formacie klasycznym

In [19]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)
circuit.draw("mpl", style="iqp", fold=-1, scale=0.6)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/b10d16cf-95da-42c0-9b47-b2e5a8516c82-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

In [20]:
slices = slice_by_gate_types(circuit)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 36 slices.


We specify the `max_error_per_slice` to be 0.005 as before. However, since the number of slices for this large-scale problem is much higher than the small scale problem, allowing an error of 0.005 per slice may end up creating a large overall backpropagation error. We can bound this by specifying `max_error_total` which bounds the total backpropagation error, and we set its value to 0.03 (which is roughly the same as in the small-scale example).

For this large-scale example, we allow a higher value for the number of commuting groups, and set it to 15.

In [21]:
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

Let us first obtain the backpropagated circuit and observable without any truncation.

In [22]:
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into {len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 7 slices.
New observable has 634 terms, which can be combined into 12 groups.
Note that backpropagating one more slice would result in 1246 terms across 27 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/164e2f00-25b6-4cf1-98f8-8b2886f007ee-1.avif" alt="Output of the previous code cell" />

Now allowing for truncation, we obtain:

In [23]:
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)

# Recombine the slices remaining after backpropagation
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs_trunc.paulis)} terms, which can be combined into {len(bp_obs_trunc.group_commuting(qubit_wise=True))} groups.\n"
    f"After truncation, the error in our observable is bounded by {metadata.accumulated_error(0):.3e}"
)
print(
    f"Note that backpropagating one more slice would result in {metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit_trunc.draw("mpl", fold=-1, scale=0.6)

Backpropagated 10 slices.
New observable has 646 terms, which can be combined into 14 groups.
After truncation, the error in our observable is bounded by 2.998e-02
Note that backpropagating one more slice would result in 1226 terms across 29 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/c05a85bc-e5ca-4e02-8c96-98b28811f335-1.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/operator-back-propagation/extracted-outputs/b444d8bc-c800-4aa3-9927-eb807e92194f-1.avif)
## Część B: Skalujemy w górę!
Użyjmy teraz propagacji wstecznej operatorów, aby zbadać dynamikę hamiltonianu 50-qubitowego łańcucha spinowego Heisenberga.

### Krok 1: Odwzorowanie klasycznych danych wejściowych na problem kwantowy
Rozważamy 50-qubitowy hamiltonian $\hat{\mathcal{H}}_{XYZ}$ dla zagadnienia w większej skali, z tymi samymi wartościami współczynników $J$ i $h$ co w przykładzie małoskalowym. Obserwabla $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$ jest również taka sama jak poprzednio. Ten problem wykracza poza możliwości klasycznej symulacji metodą brute-force.

In [24]:
# Transpile original experiment
circuit_isa = pm.run(circuit)
observable_isa = observable.apply_layout(circuit_isa.layout)

# Transpile the backpropagated experiment
bp_circuit_isa = pm.run(bp_circuit)
bp_obs_isa = bp_obs_trunc.apply_layout(bp_circuit_isa.layout)

# Transpile the backpropagated experiment with truncated observable terms
bp_circuit_trunc_isa = pm.run(bp_circuit_trunc)
bp_obs_trunc_isa = bp_obs_trunc.apply_layout(bp_circuit_trunc_isa.layout)

In [25]:
print(
    f"2-qubit depth of original circuit: {circuit_isa.depth(lambda x:x.operation.num_qubits==2)}"
)
print(
    f"2-qubit depth of backpropagated circuit: {bp_circuit_isa.depth(lambda x:x.operation.num_qubits==2)}"
)
print(
    f"2-qubit depth of backpropagated circuit with truncation: {bp_circuit_trunc_isa.depth(lambda x:x.operation.num_qubits==2)}"
)

2-qubit depth of original circuit: 48
2-qubit depth of backpropagated circuit: 40
2-qubit depth of backpropagated circuit with truncation: 36


### Step 3: Execute using Qiskit primitives

In [26]:
pubs = [
    (circuit_isa, observable_isa),
    (bp_circuit_isa, bp_obs_isa),
    (bp_circuit_trunc_isa, bp_obs_trunc_isa),
]

In [27]:
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]

estimator = EstimatorV2(mode=backend, options=options)

In [ ]:
job = estimator.run(pubs)

W przypadku tego zagadnienia w większej skali przyjęliśmy czas ewolucji równy $0.2$ z $4$ krokami Trottera. Problem dobrano tak, aby wykraczał poza możliwości klasycznej symulacji metodą brute-force, lecz mógł być symulowany metodą sieci tensorowych. Pozwala to zweryfikować wynik uzyskany poprzez propagację wsteczną na komputerze kwantowym w porównaniu z wynikiem idealnym.

Idealna wartość oczekiwana dla tego problemu, uzyskana za pomocą symulacji sieci tensorowych, wynosi $\simeq 0.89$.

In [ ]:
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

In [ ]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.7887194658035515
Backpropagated expectation value: 0.9532818300978584
Backpropagated expectation value with truncation: 0.8913400398926913


In [ ]:
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(0.89)
ax.set_ylim([0.6, 0.98])
plt.text(0.2, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/047d448f-aebf-45ff-a81b-83b2d5ca866d-1.avif" alt="Output of the previous code cell" />

Ustawiamy `max_error_per_slice` na 0.005 jak poprzednio. Jednak ponieważ liczba wycinków dla tego problemu w dużej skali jest znacznie większa niż w przykładzie małoskalowym, dopuszczenie błędu 0.005 na wycinek może prowadzić do dużego całkowitego błędu propagacji wstecznej. Możemy go ograniczyć, podając `max_error_total`, które ogranicza łączny błąd propagacji wstecznej — ustawiamy jego wartość na 0.03 (co jest mniej więcej takie samo jak w przykładzie małoskalowym).

W tym przykładzie w dużej skali dopuszczamy wyższą wartość liczby grup przemiennych i ustawiamy ją na 15.